<a href="https://www.kaggle.com/code/nagapranayimmadi/proteus-arc?scriptVersionId=325550044" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# Input data files are available in the read-only "../input/" directory
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import tkinter as tk
from tkinter import filedialog
import math
import random
from itertools import combinations
import statistics
import networkx as nx 
import matplotlib.pyplot as plt 
import scipy.io
from scipy.io import loadmat
from scipy.io.matlab import MatReadError
import mne
import copy 
import torch
import os
import torch
# 1. Install pre-compiled CPU binaries for your exact PyTorch version

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import normalized_mutual_info_score
import warnings
import numpy as np

!!pip install dcor
import dcor
print(dcor)

from scipy.signal import hilbert, butter, filtfilt, welch, iirnotch 
from IPython.display import clear_output

import tensorflow as tf

# Clears the Output of the Imports to make the notebook more clean 
clear_output() 

In [44]:
# mL imports


from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch_geometric.nn import (
    GCNConv,
    GraphConv,
    global_mean_pool,
    global_max_pool,
    global_add_pool
)

from torch.nn import Linear, ReLU
from torch_geometric.nn import Sequential, GCNConv

import torch.nn.functional as F
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)

from torch_geometric_temporal.signal import (
    StaticGraphTemporalSignal,
    DynamicGraphTemporalSignal,
    DynamicGraphStaticSignal
)

from torch_geometric_temporal.nn.recurrent import (
    GConvGRU,
    DCRNN
)

from torch_geometric.nn import MessagePassing 


# Clears the Output of the Imports to make the notebook more clean 
clear_output() 

ModuleNotFoundError: No module named 'torch_geometric'

# SEGMENT 1: FULL EDGE-BY-EDGE ALGORITHM

# RANDOMIZATION SEGMENT TO BE IMPLEMENTED LATER

# Step 1. Data Processing / Deciding which Data to use

In terms of actually processing the EEG data, removing all of the noise, etc

In [2]:
data_input = input('what data are you using? (enter "N" for new data, use "P" for premade data to test ').upper()
# factoring time out of the algorithm (TEMPORARILY MAY CHANGE)

# turning the data into dictionaries function, turning the data into dictionaires allows for easier and further processing. 
class node_to_dict: 
    def __init__(self, node_input):
        self.node_input = node_input 
    def dict(self):
        node_dict = {
            col: self.node_input[col].tolist()
            for col in self.node_input.columns }
        return node_dict


if data_input == "N": 
    # note this will only work in the API version which will be used in the interface
    # also, update this segment when you are making the inferace
    print("Now you will have to enter your data")
    def openFile():
        filepath = filedialog.askopenfilepath()
        file = open(filepath, 'r')
        reading_file = pd.read_csv('filepath')
        
    window = tk()
    button = Button(text="Open", command=openFile)
    button.pack()
    window.mainloop()

# to use the data that is already in the algorithm for testing purposes
if data_input == "P":
    print("Using the dataset for this algorithm that is in the code to test")
    # we turn these csv's into having each row and column for each node have a different list which can be iterated over
    healthy1 = pd.read_csv('/kaggle/input/datasets/nigarmahmoudshafiq/nigar-eeg-alzheimers-dataset-v1/Final dataset for the published paper/1-Healthy/CN13.csv') 
    #_________________________________________________________
    # COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
    #print(healthy1_dict) 
    # ________________________________________________________________________
    healthy : node_to_dict = node_to_dict(healthy1)
    healthy1_dict = healthy.dict()
    # print(healthy1_dict)
    moderate1 = pd.read_csv('/kaggle/input/datasets/nigarmahmoudshafiq/nigar-eeg-alzheimers-dataset-v1/Final dataset for the published paper/3-Moderate/Mod8.csv')
    mild1 = pd.read_csv('/kaggle/input/datasets/nigarmahmoudshafiq/nigar-eeg-alzheimers-dataset-v1/Final dataset for the published paper/2-Mild/Mild1.csv')
    severe1 = pd.read_csv('/kaggle/input/datasets/nigarmahmoudshafiq/nigar-eeg-alzheimers-dataset-v1/Final dataset for the published paper/4-Sever/Sv2.csv')

# to make the user know they inputted an invalid input
elif data_input != "P" and data_input !="N": 
    print("You did not enter a valid input, retry")

what data are you using? (enter "N" for new data, use "P" for premade data to test  p


Using the dataset for this algorithm that is in the code to test


# Step 1a. Using the Premade Openneuro Dataset for Processing

![](http://www.medrxiv.org/content/medrxiv/early/2024/08/06/2024.08.05.24311520/F1.large.jpg)

In [12]:
# OpenNeuro Dataset
openneuro_dataset = []
for path in range(1, 89):
    file_path = f"/kaggle/input/datasets/nagapranayimmadi/openneuro-filtered-data/sub-{path:03d}_task-eyesclosed_eeg.set"
    openneuro_dataset.append(file_path)
# ________________________________________________________________________
# COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
#print(openneuro_dataset)
# ________________________________________________________________________
# Load .set file into csv usable format

# ________________________________________________________________________
# COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
#print(x.keys())
#print(value)
# print(type(x["times"]))
# ________________________________________________________________________

x = loadmat("/kaggle/input/datasets/nagapranayimmadi/openneuro-filtered-data/sub-001_task-eyesclosed_eeg.set")
csv_openneuro_dataset = {}
# openneuro_dataframes = [] -> ARCHIVED CODE 
openneuro_patients = []
counter = 1
for csv_convert in openneuro_dataset:
    try:     
        raw = mne.io.read_raw_eeglab(csv_convert, verbose=False, preload=True)
        raw = raw.set_eeg_reference(ref_channels='average')
        # times = x["times"]
        # Extract channel/node names
        channels = raw.ch_names
        # [r[0][0] for r in raw["chanlocs"][0]]
        # times = [c[0][0] for c in x["times"][0]] -> ADD TIME TO THE DATAFRAMES AT A LATER POINT OF TIME, WHEN TEMPORAL DATA IS USED INSTEAD OF STATIC
        # Convert to CSV format 
        df = pd.DataFrame(raw.get_data().T, columns=(channels))
        del raw # now raw is taken away from the memory 
        df.drop_duplicates(inplace=True)
        # Save CSV
        df.to_csv("eeg_nodes.csv", index=False)
        # openneuro_dataframes.append(df) -> ARCHIVED CODE
        openneuro_patients.append(f"openneuro_patient{counter}")
        csv_openneuro_dataset.update({ f"openneuro_patient{counter}" : df})
        counter = counter + 1  
        # print(csv_openneuro_dataset[0].head())
    except MatReadError as e:
        pass 
# ________________________________________________________________________
# COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
# print(csv_openneuro_dataset.values())
# print(csv_openneuro_dataset)
# ________________________________________________________________________

# turning the data into dictionaries ↓↓

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


KeyboardInterrupt: 

In [21]:
"""
This segment of code helps reduce the overall memory usage in the dataframe, and allows the later code
to run significantly faster, without sacrificing accuracy
Code Taken from Source: https://www.kaggle.com/discussions/questions-and-answers/405504, individual who
uploaded = Stepan Tikhonov.
When memory optimizations are specialized and created in-depth for this algorithm, a different mode of
implementation will be utilized. 
""" 

def reduce_memory_usage(df):
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if str(col_type)[:5] == "float":
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.finfo("f2").min and c_max < np.finfo("f2").max:
                df[col] = df[col].astype(np.float16)
            elif c_min > np.finfo("f4").min and c_max < np.finfo("f4").max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)
        elif str(col_type)[:3] == "int":
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo("i1").min and c_max < np.iinfo("i1").max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo("i2").min and c_max < np.iinfo("i2").max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo("i4").min and c_max < np.iinfo("i4").max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        elif col == "timestamp":
            df[col] = pd.to_datetime(df[col])
        elif str(col_type)[:8] != "datetime":
            df[col] = df[col].astype("category")
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print("Memory usage reduced on:", round(start_mem - end_mem, 2), "Mb ", "or ", round(100 * (start_mem - end_mem) / start_mem), "%")
    return df

for df in csv_openneuro_dataset:
    optimized = reduce_memory_usage(csv_openneuro_dataset.get(df))
    csv_openneuro_dataset[df] = optimized


In [17]:
print(type(csv_openneuro_dataset.values()))

<class 'dict_values'>


In [ ]:
# OpenNeuro Dataset (BACKUP VERSION OF CODE)
openneuro_dataset = []
for path in range(1, 89):
    file_path = f"/kaggle/input/datasets/nagapranayimmadi/openneuro-filtered-data/sub-{path:03d}_task-eyesclosed_eeg.set"
    openneuro_dataset.append(file_path)
# ________________________________________________________________________
# COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
#print(openneuro_dataset)
# ________________________________________________________________________
# Load .set file into csv usable format

# ________________________________________________________________________
# COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
#print(x.keys())
#print(value)
# print(type(x["times"]))
# ________________________________________________________________________

x = loadmat("/kaggle/input/datasets/nagapranayimmadi/openneuro-filtered-data/sub-001_task-eyesclosed_eeg.set")
csv_openneuro_dataset = {}
openneuro_dataframes = []
openneuro_patients = []
counter = 1
for csv_convert in openneuro_dataset:
    try:
        x = loadmat(csv_convert, appendmat=False)
        data = x["data"]
        # times = x["times"]
        # Extract channel/node names
        channels = [c[0][0] for c in x["chanlocs"][0]]
        # times = [c[0][0] for c in x["times"][0]] -> ADD TIME TO THE DATAFRAMES AT A LATER POINT OF TIME, WHEN TEMPORAL DATA IS USED INSTEAD OF STATIC
        # Convert to CSV format 
        df = pd.DataFrame(data.T, columns=(channels))
        df.drop_duplicates(inplace=True)
        # Save CSV
        df.to_csv("eeg_nodes.csv", index=False)
        openneuro_dataframes.append(df)
        openneuro_patients.append(f"openneuro_patient{counter}")
        csv_openneuro_dataset.update({ f"openneuro_patient{counter}" : df})
        counter = counter + 1  
        # print(csv_openneuro_dataset[0].head())
    except MatReadError as e:
        pass 
# ________________________________________________________________________
# COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
# print(csv_openneuro_dataset.values())
# print(csv_openneuro_dataset)
# ________________________________________________________________________

# turning the data into dictionaires ↓↓

In [19]:
"""
Continuation of the last code segment:
A loop to turn each dataset element into a dictionary. Now, the data is processed for the next step. 
The elements of this dict are looped through, with different functions of the graph_construction class. This allows the 
individual patient based data from this dataset to be transformed into ideal forms which can be interpreted as temporal graphs
over specific snapshots. These snapshots, over a full temporal graph act as an input for the GCN.  
"""
list_of_openneuro_nodes = []
for to_dict in csv_openneuro_dataset:
    dict_key = to_dict
    function_input = csv_openneuro_dataset.get(to_dict)
    dict_key : node_to_dict = node_to_dict(function_input)
    result = dict_key.dict()
    list_of_openneuro_nodes.append(result)

# Step 1b. Function to use PLI for Node Connectivity

In [1]:
"""
PLI Quantifies the functional connectivity between 2 different nodes, which analyzes the non-zero phase differences between 2 nodes.
"""

def calculate_pli(sig1, sig2, fs=500, fmin=0.5, fmax=45.0):
    """
    this function calculates the Phase Lag Index(PLI) between two signals.
    PLI ignores zero-lag (volume conduction) correlations aswell.
    """
    
    # 1. Extracting the analytical signal using the Hilbert Transformation
    analytic_sig1 = hilbert(sig1)
    analytic_sig2 = hilbert(sig2)
    
    # 2. Extracting the instantaneous phase of both signals
    phase1 = np.angle(analytic_sig1)
    phase2 = np.angle(analytic_sig2)
    
    # 3. Calculating the phase difference between the two signals
    phase_diff = phase1 - phase2
    
    # 4. Calculate PLI:
    # The sine function forces phase differences of 0 and 180 degrees (volume conduction) to 0.
    pli_value = np.abs(np.mean(np.sign(np.sin(phase_diff))))
    return pli_value

# Step 1c. Preprocessing of Data 

In [ ]:
# Dataset Memory, RAM-Optimization, and CPU-Optimization will be implemented once the algorithm is complete.

# Full EEG Processing of the Data based on Known information, the EEG Machine, Etc.
# This will make the EEG data fully suitable for use. 
"""
Processing Steps Required based on filtering which was NOT done for the data. 
Reference for Advanced Data Processing -> https://github.com/DL4mHealth/LEAD, https://mne.tools/stable/index.html 
1. Removal of non-EEG channels
2. Notch and Band-Pass Filtering
3. Average re-referencing
4. Artifact Removal
5. Channel Alignment 
6. Frequency Alignment
7. Z-score normalization 
"""
# 1. Removal of non-EEG channels

# 2. Butterworth Filtering

butterworth_ask = input("Is your data butterworth-filtered? (Y/N) ").upper()
if input == N:
    # butterworth filtering
else: 
    print("Moving onto the next filtering round!")

# 3. Notch and Band-Pass Filtering
    
notch_ask = input("Is your data notch-filtered? (Y/N) ").lower()
if input == N:
    # filtering
else: 
    print("Moving onto the next filtering round!")

bandpass_ask = input("Is your data band-pass filtered? (Y/N) ").upper()
if input == N:
    # filtering
else: 
    print("Moving onto the next filtering round!")
    
# 4. Average re-referencing
rerefavg_ask = input("Has there been avg re-referencing done on your data? (Y/N) ").upper()
if input == N:
    # filtering
else: 
    print("Moving onto the next filtering round!")
    
# 5. Artifact Removal
artifact_ask = input("Has there been artifact removal from your data? (Y/N) ").upper()
if input == N:
    # filtering
else: 
    print("Moving onto the next filtering round!")
    
# 6. Channel Alignment 
chosen_columns = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]

# if data.column is not in chosen_columns:
    # remove that column

# 7. Frequency Alignment
freqalign_ask = input("Has there been frequency alignment done on your data? (Y/N) ").upper()
if input == N:
    # filtering
else: 
    print("Moving onto the next filtering round!")
    
# 8. Z-score normalization 
zscorenorm_ask = input("Has there been z-score normalization done on your data? (Y/N) ").upper() 
if input == N:
    # filtering
else: 
    print("Moving onto the next filtering round!")

print("The data is now fully filtered!")

# Step 2. General Functions to Help with Graph(Node-Edge type Graph) Construction

<p align="center">
  <img src="https://i.sstatic.net/ZXXjJ.png" width="50%">
</p>

In [2]:
class graph_construction: 
    # These are the class-based variables that are appended with the key important features required for these functions. 
    connectivity_dict = {} # This is for creating the connectivity dict in the form { edge(such as Fp1, Fp2) : Edge Value}
    edgetoedge_dict = {} 
    parent_node_features_dict = {}
    timestamp_wise_node_features = {}

    def __init__(self, node_list): 
        self.node_list = node_list 
        self.list_of_nodes = list(node_list)
        self.df = pd.DataFrame(columns=self.list_of_nodes, index=self.list_of_nodes)
        self.node_list = copy.deepcopy(node_list)
        self.chunks_length = None
    
    def create_edges(self):
        """
        Divide the dataset into individual components like [ list_of_openneuro_nodes[0] ] , and then from there, we transform
        the data by keeping the keys the same, but dividing the values an x number of times, while keeping the keys the same
        . This is to get temporal lists of the edges created over time from the nodes values, and also get the Distance
        Correlation values in def distance_correlation, or any other edge connectivity values which require a temporal aspect. 
        """
        for temporal_nodes in self.node_list: 
            node_pre = self.node_list.get(temporal_nodes)
            chunks = [node_pre[x:x+2000] for x in range(0, len(node_pre), 2000)]
            self.node_list[temporal_nodes] = chunks
        self.chunks_length = len(chunks)
        # ________________________________________________________________________
        # COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
        #print(self.node_list["Fp1"][0:2])
        # list_of_openneuro_nodes[0]
        # ________________________________________________________________________
        for node1, node2 in combinations(self.node_list, 2): # the larger loop for the entirety of all edge connections 
            z_list = [] 
            z_means = []
            for length in range (0, self.chunks_length, 1):
                # Pearson Connectivity Formula
                # key for the key-value pair of connectivity values
                key_of_connectivity = (str(node1), str(node2))
                # here we add it to be the first temporal segment being done for, and it keeps going (NEED TO ADD) -> another loop needs to be added for
                # individual edge creation inside this segment 
                # x = (self.node_list.get(node1)[length])
                # y = (self.node_list.get(node2)[length])
                x = (self.node_list[node1][length])
                y = (self.node_list[node2][length])
                z = calculate_pli(x, y, fs=500)
                z_list.append(z) 
            # graph_construction.connectivity.append(z) -> ARCHIVED CODE
            # creating the connectivity dictionary
            graph_construction.connectivity_dict.update({key_of_connectivity : z_list}) 
            # self.df.loc[node1, node2] = z -> add back now
            # self.df.loc[node2, node1] = z -> add back now
        # temporal adjacency matrices -> example for one edgee in between nodes, there is a list of values
        #pd.set_option('future.no_silent_downcasting', True)-> add back now
        #self.df = self.df.replace(np.nan,0) -> add back now

    def node_values_creation(self): #this part is for getting features, aka x 
        """
        The node features created in this segment for each timeframe are:
        1. Mean Voltage 
        2. Band Frequency 
            - Alpha Power
            - Beta Power
            - Theta Power
            - Delta Power
            - Gamma Power
        3. Variance
        Additionally, here in node_values_creation, we will also get the chunkwise node features. 
        """
        
        # looping chunkwise
        for chunk in range(0, self.chunks_length, 1):
            # finding out these values for each individual node of the chunk 
                # creating the average node list -> this fixes the potential A1/A2 mastoid reference in the Openneuro Dataset
            
            # for each node in each chunk, to apply set_eeg_reference to them.
            for values in self.node_list.keys(): 

                # 1. for node mean values over time
                node_mean = statistics.mean(self.node_list[values][chunk]) 
                
                # 2. PSD and Band Frequencies
                # saving all of the band frequencies 
                chunked_node_features = []
                alpha = []
                beta = []
                theta = []
                delta = []
                gamma = []
                
                # for PSD(power spectral densities) over time
                freqs, psd = welch(np.array(avg_node_list[values][chunk]), fs=500, nperseg=2000)
                # ________________________________________________________________________
                # COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
                # print(psd)
                # ________________________________________________________________________
                
                # Calculating the band frequencies from the PSD values   
                delta_mask = (freqs >= 0.5) & (freqs < 4)
                theta_mask = (freqs >= 4) & (freqs < 8)
                alpha_mask = (freqs >= 8) & (freqs < 13)
                beta_mask  = (freqs >= 13) & (freqs < 30)
                gamma_mask = (freqs >= 30) & (freqs < 45)

                # Calculating the means of the Frequencies bands(if present) to act as node features
                # the np.trapezoid is meant to find the area itself under the graph, which is much more reliable for the band frequency
                #
                total_mask = (freqs >= 0.5) & (freqs < 4)
                total_band_power = np.trapezoid(psd[total_mask], freqs[total_mask])
                if any(alpha_mask):
                    alpha_band_power = np.trapezoid(psd[alpha_mask], freqs[alpha_mask])
                    alpha_rbp = alpha_band_power / total_band_power
                if any(beta_mask):
                    beta_band_power = np.trapezoid(psd[beta_mask], freqs[beta_mask]) # first variable for RBP(Relative Band Power)
                    beta_rbp = beta_band_power / total_band_power
                if any(theta_mask):
                    theta_band_power = np.trapezoid(psd[theta_mask], freqs[theta_mask])
                    theta_rbp = theta_band_power / total_band_power
                if any(delta_mask):
                    delta_band_power = np.trapezoid(psd[delta_mask], freqs[delta_mask])
                    delta_rbp = delta_band_power / total_band_power
                if any(gamma_mask):
                    gamma_band_power = np.trapezoid(psd[gamma_mask], freqs[gamma_mask])
                    gamma_rbp = gamma_band_power / total_band_power

                # ------ print statements for testing ------------- 
                print(f"-------------- CHUNK {chunk} : NODE {values} ------------------")
                print("Relative Alpha Band Power: ", alpha_rbp)
                print("Relative Beta Band Power: ", beta_rbp)
                print("Relative Theta Band Power: ", theta_rbp)
                print("Relative Delta Band Power: ", delta_rbp)
                print("Relative Gamma Band Power: ", gamma_rbp)

                # 3. for node variance over time
                node_variance = statistics.variance(self.node_list[values][chunk])
                # ________________________________________________________________________
                # COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
                # print(node_variance)
                # ________________________________________________________________________

                # all of the node features for one node, chunkwise 
                all_node_features = node_mean, alpha_rbp, beta_rbp, theta_rbp, delta_rbp, gamma_rbp, node_variance
                chunked_node_features.extend(all_node_features)
                
                graph_construction.parent_node_features_dict.update({ f"{values} for chunk {chunk}" : chunked_node_features})
            graph_construction.timestamp_wise_node_features.update({f"All node values for chunk {chunk}" : graph_construction.parent_node_features_dict})
            # print(graph_construction.timestamp_wise_node_features["All node values for chunk 0"])
            # print(self.node_list["Fp1"][0:2])

    def distance_correlation(self): # this part creates the distance correlation
        for edge1, edge2 in combinations(graph_construction.connectivity_dict, 2):
            total_results = []
            edge_keys = []
            for length_dcor in range (0, 20, 1): # self.chunks_length
                value_a = np.array(graph_construction.connectivity_dict.get(edge1)[length_dcor])
                value_b = np.array(graph_construction.connectivity_dict.get(edge2)[length_dcor])
                result = dcor.distance_correlation(value_a, value_b)
                edge_key = (edge1, edge2)
                edge_keys.append(edge_key)
                total_results.append(result)
            graph_construction.edgetoedge_dict.update({ edge_keys : total_result})
        print("-----TOTAL DCOR VALUES OVER CHUNKS-----")
        print(graph_construction.edgetoedge_dict)
        # print(graph_construction.connectivity_dict) -> WILL ONLY BE UNCOMMENTED FOR TESTING PURPOSES  
        # Need to make DCOR temporal

In [3]:
"""
Overall Data Present
1. Node Features
2. Edge Values
3. Distance Correlation
""" 
# To create the list of graphs 
openneuro_graph_list = []
for list_main in list_of_openneuro_nodes:
    patientval = graph_construction(list_main)
    patientval.create_edges()
    patientval.node_values_creation()
    patientval.distance_correlation()
    break
    
    # This is the defined graph structure as an input we will need to keep for the GCN 
    # Note, tensors should be looked at vertically(where each column is what you are looking for)
    # Make this into a loop so all of the data till now can be used this way
    dataset = StaticGraphTemporalSignal(
        features = features,            # List of NumPy arrays, one per time window, each shape [num_nodes, num_node_features]
        edge_index = edge_index,        # NumPy array, shape [2, num_edges], which is fixed, same every snapshot but the values are not fixed
        edge_weight = edge_weight,      # NumPy array, shape [num_edges], Pearson values, fixed or per snapshot but are not fixed values wise
        distance_correlation_index = distance_correlation_index, # connectivity metric between the edges index
        distance_correlation_weight = distance_correlation_weight  # connectivity metric between the edges weight
        # Targets are seperate and will be incorporated in the GCN segment with the loss function later on  
    )
    openneuro_graph_list.append(dataset)

# possibillity for current error -> self.node_list is getting mutated

NameError: name 'list_of_openneuro_nodes' is not defined

In [42]:
print(type(list_of_openneuro_nodes[0]))

<class 'dict'>


In [13]:
print(patientval.chunks_length)

150


In [ ]:
# This is the code segment where all of the values for the StaticGraphTemporalSignal object are created. 

# ____________________________________

# Nodes 

# ____________________________________

# For nodes, most of the heavy-lifting is done in the graph_construction class through the methods. 

# ____________________________________

# Edges

# ____________________________________

# defining all of the existing nodes 
nodes = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz", "C4", "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2"]

edge_index_dict = {}
# making a loop to create the list of nodes and their value's which they are assigned, which applies throughout the algorithm
counter = 0 
for key_assignment in nodes:
    edge_index_dict.update({ key_assignment : counter})
    counter = counter + 1
# ________________________________________________________________________
# COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
# print("------THE DICTIONARY OF NUMERICAL VALUES CONNECTED TO NODES------")
# print(edge_index_dict)
# ________________________________________________________________________

# getting the edge index (unordered)
edge_index = []
for edge_indice1, edge_indice2 in combinations(edge_index_dict.values(), 2):
    edge_indexxed_xandy = (edge_indice1, edge_indice2)
    edge_indexxed_yandx = (edge_indice2, edge_indice1)
    edge_index.append(edge_indexxed_xandy)
    edge_index.append(edge_indexxed_yandx)

# normal edge index 
normal_edge_index = edge_index 

# print("------INDEX OF EDGES------")
# print(normal_edge_index)

# altered edge index(unordered) (need to merge with the loop later, so that the loop turns the values into tensors, and 
# then transposes them)
edge_index = torch.tensor(edge_index)
edge_index = torch.t(edge_index)
# print(edge_index)
# print(edge_index.shape) -> the shape is proper as [2, ]

# creating edge connecitivity strings
edges = []
for node_unit1, node_unit2 in combinations(nodes, 2 ):
    edge1 = (node_unit1, node_unit2)
    edge2 = (node_unit2, node_unit1)
    edges.append(edge1)
    edges.append(edge2)
# print("-----EDGE-TO-EDGE CONNECTION VALUES-----")
# print(edges)

# creating a combined dict for them to later be connected to connectivity_dict to produce edge_index and edge_values
# print("-----COMBINED DICT OF BOTH-----")
edge_connectivity_index = dict(zip(edges, normal_edge_index))
# print(edge_connectivity_index)

# GOAL - TO PROPERLY MAP EDGE INDEX AND EDGE WEIGHTS TO EACHOTHER. 
ticker = 0 
edge_index_ordered = []
edge_weights = []
timestamps_of_edge_weights = {}
for weight_chunks in range(0, patientval.chunks_length, 1):
    for keys in graph_construction.connectivity_dict:
        if weight_chunks == 0:
            edge_index_ordered.append(edge_connectivity_index[keys])
        edge_weights.append(graph_construction.connectivity_dict[keys][weight_chunks])
        timestamps_of_edge_weights.update({f"this is chunk {ticker} with the edge weights:" : edge_weights})
        ticker = ticker + 1 
print("-----EDGE INDEX-------\n", edge_index_main) 
print("-----EDGE WEIGHTS-------\n", timestamps_of_edge_weights)

# ____________________________________

#Distance Correlation 

# ____________________________________

edges = edges # just to assossiate 

# getting the dcor index(unordered)
print(edge_index)
edge_index = []
for edge_indice1, edge_indice2 in combinations(edge_index_dict.values(), 2):
    edge_indexxed_xandy = (edge_indice1, edge_indice2)
    edge_indexxed_yandx = (edge_indice2, edge_indice1)
    edge_index.append(edge_indexxed_xandy)
    edge_index.append(edge_indexxed_yandx)


# Getting the corrosponding dcor values which can equal the dcor values of the edgetoedge_dict defined in the
# distance correlation function 
dcor_values = []
for dcor_indice1, dcor_indice2 in combinations(edges, 2):
    dcor_indexxed_xandy = (dcor_indice1, dcor_indice2)
    dcor_indexxed_yandx = (dcor_indice2, dcor_indice1)
    dcor_values.append(edge_indexxed_xandy)
    dcor_values.append(edge_indexxed_yandx)
print(dcor_values)

# GOAL - TO PROPERLY MAP DCOR INDEX AND DCOR WEIGHTS TO EACHOTHER. 
score = 0 
dcor_index_ordered = []
dcor_weights = []
timestamps_of_dcor_weights = {}
for weight_chunks in range(0, patientval.chunks_length, 1):
    for keys in dcor_values:
        if weight_chunks == 0:
            dcor_index_ordered.append(edge_connectivity_index[keys]) # dont have an equivalent yet need to update
        dcor_weights.append(graph_construction.edgetoedge_dict[keys][weight_chunks])
        timestamps_of_edge_weights.update({f"this is chunk {score} with the dcor weights:" : dcor_weights})
        score = score + 1 
print("-----DCOR INDEX-------\n", edge_index_main) 
print("-----DCOR WEIGHTS-------\n", timestamps_of_edge_weights)

graph_construction.edgetoedge_dict.update({ edge_keys : total_result})

In [ ]:
# -------- ARCHIVED CODE ---------- 
# duplicate of edge creation function and graph creation function and more incase anything happens to the main one

 def create_edges(self):
        """
        here, the data starts to become temporal
        We divide the dataset into its individual components like [ list_of_openneuro_nodes[0] ] , and then from there, we transform
        the data by keeping the keys the same, but dividing the values x number of times while keeping the keys the same
        so that we can get temporal lists of the edges created over time from the nodes values changing over time, and also get the 
        MI(Mutual Information) values, or any other edge connectivity values which require a temporal aspect. 
        """
        # list_of_openneuro_nodes[0]
        for node1, node2 in combinations(self.node_list, 2): 
            # Pearson Connectivity Formula
              # key for the key-value pair of connectivity values
            key_of_connectivity = (node1, node2)
            graph_construction.normal_key.append(key_of_connectivity)
            x = (self.node_list.get(node1))
            y = (self.node_list.get(node2))
            z = statistics.correlation(x, y)
            graph_construction.connectivity.append(z)
            # creating the connectivity dictionary
            graph_construction.connectivity_dict.update({key_of_connectivity : z})
            # creating the adjacency matrix in the loop itself 
            self.df.loc[node1, node2] = z 
            self.df.loc[node2, node1] = z
        # temporal adjacency matrices -> example for one edgee in between nodes, there is a list of values
        pd.set_option('future.no_silent_downcasting', True)
        self.df = self.df.replace(np.nan,0) 


    def graph_creation(self):
        # G is initialized here.
        G = nx.from_pandas_adjacency(self.df)
        nx.set_node_attributes(G, self.node_values, "mean_voltage")
        # return G 

    def node_values_creation(self): #this part is for getting features, aka x 
        for values in self.node_list.keys() :
            node_mean = statistics.mean(self.node_list.get(values))
            node_value = values
            graph_construction.node_values.update({ node_value : node_mean })
            # ________________________________________________________________________
            # COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
            #print(node_values) 
            # ________________________________________________________________________

# Graph Visualization -> Which can be done for every single graph 
# NOTE: This is only needed for some specific testing outcomes

nx.draw(
    openneuro_graph_list.get("openneuro_patient1"),
    with_labels = True,
    node_color ='skyblue',
    edge_color ='black',
    width = 0.8,
    font_size = 6
)
plt.show()

# DYNAMIC GRAPH - FOR LATER CODE
# Multiple edge values for the same node connections need to be analyzed over time to make the data dynamic instead of static
# for time, what needs to be analyzed:
# 1. Sampling Rate in Hz - Cycles Per Second - Here it would be 250Hz of Voltage
# 2. Recording Duration - Number of Columns / Sampling Rate 
# 3. Number of Columns - Here it would be 1024 
sampling_rate = int(input("enter the sample rate(in Hz): "))
columns_number = len(healthy1_dict.get("Fp1"))
recording_duration = columns_number/sampling_rate
print(recording_duration, "seconds is the amount of time the test was done for!")
# EDIT THIS LATER


                # finding the total useful PSD to calculate the relative band power
                useful_psd = alpha_psd + beta_psd + theta_psd + delta_psd + gamma_psd

                # calculating relative band power
                if alpha_psd is not None:
                    alpha_psd = alpha_psd / useful_psd
                if beta_psd is not None:
                    beta_psd = beta_psd / useful_psd
                if theta_psd is not None:
                    theta_psd = theta_psd / useful_psd
                if delta_psd is not None:
                    delta_psd = delta_psd / useful_psd
                if gamma_psd is not None:
                    gamma_psd = gamma_psd / useful_psd


# SEGMENT 2: THE GCN
May add a transformer in the layer if the requirements of the core model are met

In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

# dataset = Planetoid(root='data/Planetoid', name='Cora', transform=NormalizeFeatures())
# print(dataset[0])

In [ ]:
random_seed = 43 # Will change later to being multiple seeds with are averaged
torch.manual_seed(1234567)
seed_everything(random_seed)
plt.style.use('dark_background')
accuracy_list = []

"""
1. Node and Edge Graph -> With Node & Edge Values
Basically the input can be all of the voltages of the nodes  
The output is all of the new and transformed node features based on what is being done
These new representations will all be analyzed by an MLP which will give them an importance weight
Then, at the end, a simple algorithm will compile the importance weight for all the 4 different edge factors, and determine whether the person has alzheimer's 
"""

In [ ]:
"""
3 things for the data, but 2 of them are in the graph.
1. node means and edge values
2. labels which they can be assigned to
"""
dataset = d

In [ ]:
# Step 2. Creating the Model
"""
Note, this segment of the model is rather adaptable based on the model, as the base of the model is based off PyTorch Geometric. However, this model
has the most advanced infrastructure for the function(or atleast one of the highest), it is possible to edit this model for future approaches in 
the concept aswell. Though in due time this model will become very specialized, but this version will remain as archived and will also be opensource
Pytorch GCN Fundamental Conv Layer: https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.GCNConv.html?utm_source=chatgpt.com#torch_geometric.nn.conv.GCNConv.backward 
Pytorch GNN's Implementation: https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#torch_geometric.nn.conv.GCNConv
"""

#Creating GCNConvolutional as a Child class of GCNConv which will have custom properties that will allow the Algorithm to function
class GCNConvolutional(GCNConv):
    def propogate(edge_index, x=x, edge_weight=edge_weight, distance_correlation=distance_correlation)
    def message(x_j: Tensor):
        return edge_weight.view(-1, 1) * x_j as x_i 
        edge_to_edge = message(x_i: Tensor) # 30% weightage
        return something something * x_i 
    def aggregate():
    def update(): 
        
# inherit from GCNConv itself, and edit messagepassing through making it a function
    
# Convoltuional Layer is GCNConv
model = Sequential(
    'x, edge_index', [
    # layer 1 or first propogation
    (GCNConvolutional(in_channels, 64, ADD_AGGREGGATION_LAYER), 'x, edge_index -> x'),

    # ReLu activation function 
    ReLU(inplace=True),

    # layer 2 on second propogation 
    (GCNConvolutional(64, 64, ADD_AGGREGGATION_LAYER), 'x, edge_index -> x'),

    # ReLu activation function 
    ReLU(inplace=True),

    # Applying Lineararity to the Results 
    Linear(64, out_channels),
]
                  )

x = self.gcn(...)

x = F.relu

x = self.gcn(...)

x = F.relu

x = self.gru(...)

x = global_mean_pool(x, batch)

x = self.classifier(x)

# How to know what type of severeity of alzheimer's the results indicate to? -> classify the data to a score

# The results will be converted into an MMSE Score which is a helper method to help diagnose alzheimer's 

# The scale for the classes is a minimum of 0, and a maximum of 30 based on an MMSE score
classes = {
    "healthy" : (27, 30),
    "mild" : (21, 26),
    "moderate" : (10, 20),
    "severe" : (0, 9)
}
y_dict = {}
for subject in csv_openneuro_dataset:
    print(subject)
y_dict = {
    "sub-001": 16,
    
if classes > 1 or < 0:
    print(str( KeyError('The computations have been resulted in either being less than zero, or one, which does not work on the scale')))

In [ ]:
# GRU Model 
"""
Note, for this version, the GRU model is taken from the following github.
https://github.com/lehaifeng/T-GCN 
"""

import argparse
import torch
import torch.nn as nn


class GRULinear(nn.Module):
    def __init__(self, num_gru_units: int, output_dim: int, bias: float = 0.0):
        super(GRULinear, self).__init__()
        self._num_gru_units = num_gru_units
        self._output_dim = output_dim
        self._bias_init_value = bias
        self.weights = nn.Parameter(
            torch.FloatTensor(self._num_gru_units + 1, self._output_dim)
        )
        self.biases = nn.Parameter(torch.FloatTensor(self._output_dim))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weights)
        nn.init.constant_(self.biases, self._bias_init_value)

    def forward(self, inputs, hidden_state):
        batch_size, num_nodes = inputs.shape
        # inputs (batch_size, num_nodes, 1)
        inputs = inputs.reshape((batch_size, num_nodes, 1))
        # hidden_state (batch_size, num_nodes, num_gru_units)
        hidden_state = hidden_state.reshape(
            (batch_size, num_nodes, self._num_gru_units)
        )
        # [inputs, hidden_state] "[x, h]" (batch_size, num_nodes, num_gru_units + 1)
        concatenation = torch.cat((inputs, hidden_state), dim=2)
        # [x, h] (batch_size * num_nodes, gru_units + 1)
        concatenation = concatenation.reshape((-1, self._num_gru_units + 1))
        # [x, h]W + b (batch_size * num_nodes, output_dim)
        outputs = concatenation @ self.weights + self.biases
        # [x, h]W + b (batch_size, num_nodes, output_dim)
        outputs = outputs.reshape((batch_size, num_nodes, self._output_dim))
        # [x, h]W + b (batch_size, num_nodes * output_dim)
        outputs = outputs.reshape((batch_size, num_nodes * self._output_dim))
        return outputs

    def hyperparameters(self):
        return {
            "num_gru_units": self._num_gru_units,
            "output_dim": self._output_dim,
            "bias_init_value": self._bias_init_value,
        }


class GRUCell(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super(GRUCell, self).__init__()
        self._input_dim = input_dim
        self._hidden_dim = hidden_dim
        self.linear1 = GRULinear(self._hidden_dim, self._hidden_dim * 2, bias=1.0)
        self.linear2 = GRULinear(self._hidden_dim, self._hidden_dim)

    def forward(self, inputs, hidden_state):
        # [r, u] = sigmoid([x, h]W + b)
        # [r, u] (batch_size, num_nodes * (2 * num_gru_units))
        concatenation = torch.sigmoid(self.linear1(inputs, hidden_state))
        # r (batch_size, num_nodes * num_gru_units)
        # u (batch_size, num_nodes * num_gru_units)
        r, u = torch.chunk(concatenation, chunks=2, dim=1)
        # c = tanh([x, (r * h)]W + b)
        # c (batch_size, num_nodes * num_gru_units)
        c = torch.tanh(self.linear2(inputs, r * hidden_state))
        # h := u * h + (1 - u) * c
        # h (batch_size, num_nodes * num_gru_units)
        new_hidden_state = u * hidden_state + (1 - u) * c
        return new_hidden_state, new_hidden_state

    @property
    def hyperparameters(self):
        return {"input_dim": self._input_dim, "hidden_dim": self._hidden_dim}


class GRU(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, **kwargs):
        super(GRU, self).__init__()
        self._input_dim = input_dim  # num_nodes for prediction
        self._hidden_dim = hidden_dim
        self.gru_cell = GRUCell(self._input_dim, self._hidden_dim)

    def forward(self, inputs):
        batch_size, seq_len, num_nodes = inputs.shape
        assert self._input_dim == num_nodes
        outputs = list()
        hidden_state = torch.zeros(batch_size, num_nodes * self._hidden_dim).type_as(
            inputs
        )
        for i in range(seq_len):
            output, hidden_state = self.gru_cell(inputs[:, i, :], hidden_state)
            output = output.reshape((batch_size, num_nodes, self._hidden_dim))
            outputs.append(output)
        last_output = outputs[-1]
        return last_output

    @staticmethod
    def add_model_specific_arguments(parent_parser):
        parser = argparse.ArgumentParser(parents=[parent_parser], add_help=False)
        parser.add_argument("--hidden_dim", type=int, default=64)
        return parser

    @property
    def hyperparameters(self):
        return {"input_dim": self._input_dim, "hidden_dim": self._hidden_dim}

In [ ]:
# Step 4. Training

In [ ]:
# Step 5. Validation

In [ ]:
# Step 6. Testing 

# SEGMENT 3: INFERENCE(ALZHEIMER'S PROBABILITY FOR JUST ONE PERSON) 
80% Weightage

# SEGMENT 4 - AI-BASED ALZHEIMER'S ANALYSIS BASED ON QUESTIONAIRE
20% WEIGHTAGE

In [ ]:
# Imports

In [ ]:
# Code

# SEGMENT 5 - FINAL COMPILER (DETERMINES WHETHER THE INDIVIDUAL HAS ALZHEIMER'S

In [ ]:
"""
This is the segment of the algorithm which will determine whether the individual has alzheimer's or a neurodegenerative condition in general.
In the first iteration it will give a determination, and will be coded to also provide a confidence rating. This segment will take as input
1. The EEG Segment -> The output from this segment is taken into account as 80% of the weightage"
2. The Questionaire -> The output from this segment is taken into account as 20% of the weightage
"""

In [ ]:
# this code is for keeping the cell continously active to bypass the 40-minute idle timeout
import time
try:
    while True:
        time.sleep(60)  # Pauses for 1 minute
except KeyboardInterrupt:
    print("Loop stopped manually.")